# Money Leak Backtest — SEITH (research isolated)
Plotly offline fig.show() — 7 cells, no apps/kronos-sidecar pollution (grep plotly →0).
Data: universe-100.json (100 stratified) → Sectors /v2/daily/{symbol}/ (fallback tests/fixtures/bbca-ohlcv-400.json) → normalize (OHLC→excluded/volume→0/median) → Kronos 400→20 (fallback kronos-pred-20.json) → Score 30/20/30/20 → Flag |Z|>2 → Rank → backtest-100.json

In [ ]:
import json
import pathlib
uni = json.loads(pathlib.Path('universe-100.json').read_text())
assert len(uni['items']) == 100, len(uni['items'])
assert uni['stratified'] == {'FINANCE':25,'ENERGY':20,'CONSUMER':20,'INFRA':20,'OTHER':15}
print(f"universe {uni['as_of']} {len(uni['items'])} OK")

In [ ]:
# 02 normalize + sector-median fallback per market (Id vs Sg terpisah)
import json
import pathlib
med = json.loads(pathlib.Path('../tests/fixtures/sector-median.json').read_text())
ill = json.loads(pathlib.Path('../tests/fixtures/illiquid-ohlcv.json').read_text())
print('sector-median', list(med.get('id',{}).keys())[:3], 'illiquid', len(ill))
# OHLC 0/nan → excluded, volume None→0, rasio None→median, lookback>512→422 (see normalize.rs)

In [ ]:
# 03 Kronos 400→20 zero-shot (KRONOS_MOCK=1 for CI, real predict_batch T1.0 top_p0.9 max_context 512)
import json
import pathlib
pred = json.loads(pathlib.Path('../tests/fixtures/kronos-pred-20.json').read_text())
print('kronos-pred-20', len(pred), pred[0] if pred else 'empty')
# live: POST :8001/predict_batch {market, df, x_timestamp, y_timestamp, pred_len:20, T:1.0, top_p:0.9}

In [ ]:
# 04-05 Score 0-100 (30ER+20(100-|Z|)+30QV+20SM clamp) + Flag |Z|>2 OR vol>2σ → Rank
import json
import pathlib
bt = json.loads(pathlib.Path('backtest-100.json').read_text())
assert len(bt['items']) == 100
top5 = sorted([x for x in bt['items'] if x['anomaly']['flag']], key=lambda x: abs(x['anomaly']['z']), reverse=True)[:5]
print('backtest', bt['as_of'], 'items', len(bt['items']), 'top5 flags', len(top5), 'metrics', bt['metrics'])
# rank Mispricing desc → |Z| tie-break; anomaly sort |Z| desc → score tie-break (see ranking/service.rs)

In [ ]:
# 06 metrics (Sharpe/maxDD/win rate) — verifiable
import json
m = json.loads(open('backtest-100.json').read())['metrics']
print({k: round(float(v),4) for k,v in m.items() if isinstance(v,(int,float))})
assert 0 <= m['hit_rate'] <= 1
assert m['drawdown'] <= 0

In [ ]:
# 07 equity vs IHSG plot (plotly 5.* isolated, offline)
try:
    import plotly
    import plotly.graph_objects as go
    import json
    bt = json.loads(open('backtest-100.json').read())
    ec = bt.get('equity_curve', [])
    fig = go.Figure()
    if ec:
        fig.add_trace(go.Scatter(x=[r['date'] for r in ec], y=[r['return'] for r in ec], name='SEITH'))
        fig.add_trace(go.Scatter(x=[r['date'] for r in ec], y=[r['bench'] for r in ec], name='IHSG'))
    fig.update_layout(title='Equity vs IHSG (research isolated)', template='plotly_dark')
    fig.show()
    print('plotly', plotly.__version__, 'traces', len(fig.data))
except Exception as e:
    print('plotly fallback (no display):', e)
    print('equity_curve len', len(json.loads(open('backtest-100.json').read()).get('equity_curve',[])))


In [ ]:
# 08 Table 100 — ranking verifiable (sort Score desc)
import json, pathlib
bt = json.loads(pathlib.Path("backtest-100.json").read_text(encoding="utf-8"))
items = sorted(bt["items"], key=lambda x: x["mispricingScore"], reverse=True)
assert len(items) == 100, len(items)
rows = []
for r in items:
    rows.append({"ticker": r["ticker"], "sector": r["sector"], "close": r["close"], "mispricingScore": r["mispricingScore"], "ER": r.get("kronos",{}).get("forecastReturn", r.get("components",{}).get("expected_return")), "|Z|": abs(r.get("anomaly",{}).get("z",0)), "QV": r.get("components",{}).get("quality_value"), "SM": r.get("components",{}).get("sector_mom"), "flag": r.get("anomaly",{}).get("flag"), "rank": r.get("rank"), "research_source": r.get("research",{}).get("source","")})
try:
    import pandas as pd
    df = pd.DataFrame(rows)
    print(df.head(20).to_string(index=False))
except Exception:
    for row in rows[:20]:
        print(row)
print(f"rows {len(rows)} display 20/100 OK")
assert len(rows)==100


In [ ]:
# 09 Fig 2x2 plotly_dark — hist ER + scatter ER vs |Z| + stacked Top-20 + heatmap 10x10
import plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json, pathlib
bt = json.loads(pathlib.Path("backtest-100.json").read_text(encoding="utf-8"))
sc = json.loads(pathlib.Path("scores_98.json").read_text(encoding="utf-8"))
ers = [x["er"] for x in sc["items"]]
zs = [abs(x["z"]) for x in sc["items"]]
top20 = sorted(bt["items"], key=lambda x: x["mispricingScore"], reverse=True)[:20]
fig = make_subplots(rows=2, cols=2, subplot_titles=("Hist ER 98","ER vs |Z| 98","Stacked Top-20 4 segs","Heatmap 10x10"))
fig.add_trace(go.Histogram(x=ers, nbinsx=20, name="ER hist", marker_color="#fbbf24"), row=1, col=1)
fig.add_trace(go.Scatter(x=ers, y=zs, mode="markers", name="ER vs |Z|", marker=dict(color=["#ef4444" if abs(x["z"])>2 else "#10b981" for x in sc["items"]], size=6)), row=1, col=2)
for k,col in [("expected_return","#fbbf24"),("anomaly_z","#ef4444"),("quality_value","#10b981"),("sector_mom","#3b82f6")]:
    fig.add_trace(go.Bar(x=[r["ticker"] for r in top20], y=[r["components"].get(k,0) for r in top20], name=k, marker_color=col), row=2, col=1)
fig.add_trace(go.Heatmap(z=[[r["mispricingScore"] for r in bt["items"][i*10:(i+1)*10]] for i in range(10)], colorscale="Viridis", showscale=False), row=2, col=2)
fig.update_layout(template="plotly_dark", barmode="stack", height=700, title="SEITH 2x2 — ER hist + scatter + stacked + heatmap", margin=dict(l=40,r=40,t=60,b=40))
fig.show()
print("plotly", plotly.__version__, "traces", len(fig.data))
assert len(fig.data) >= 4


In [ ]:
# 10 Memos Top-10 — Fund/Tech/Synth + pie llm 10 vs template 90
import plotly
import plotly.graph_objects as go
import json, pathlib
bt = json.loads(pathlib.Path("backtest-100.json").read_text(encoding="utf-8"))
memos = json.loads(pathlib.Path("memos_top10.json").read_text(encoding="utf-8")).get("memos",{})
llm_n = sum(1 for x in bt["items"] if x.get("research",{}).get("source")=="llm")
tmpl_n = 100 - llm_n
for t in sorted(memos.keys(), key=lambda k: memos[k].get("rank",99))[:10]:
    f = memos[t].get("fundamental_memo","")[:300]
    tec = memos[t].get("technical_memo","")[:300]
    syn = memos[t].get("synthesizer_memo","")[:300]
    print(f"[{memos[t].get('rank')}] {t} — Fund: {f} | Tech: {tec} | Synth: {syn}")
    print("Bukan rekomendasi investasi. Informasi & analisis saja.")
    print("---")
fig2 = go.Figure(data=[go.Pie(labels=["llm","template"], values=[llm_n, tmpl_n], hole=.4, marker_colors=["#fbbf24","#27272a"])])
fig2.update_layout(template="plotly_dark", title=f"Research source — llm {llm_n} vs template {tmpl_n}")
fig2.show()
print(f"pie llm {llm_n} vs template {tmpl_n} — memos {len(memos)}")
assert llm_n==10 and tmpl_n==90
assert len(memos)==10
